<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 19

<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Subscription в C#, который будет представлять подписки на
различные услуги. На основе этого класса разработать 2-3 производных класса,
демонстрирующих принципы наследования и полиморфизма. В каждом из классов
должны быть реализованы новые атрибуты и методы, а также переопределены
некоторые методы базового класса для демонстрации полиморфизма.


#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [5]:
public class Subscription
{ 
    private int _subscriptionId{get; set;}
    private string _serviceName{get; set;}
    private decimal _cost{get; set;}

    public int SubscriptionId
    {
        get => _subscriptionId;
        set
        {
            if (value <= 0) throw new ArgumentException("ID должен быть больше нуля.");
            _subscriptionId = value;
        }
    }

    public string ServiceName
    {
        get => _serviceName;
        set
        {
            if (string.IsNullOrWhiteSpace(value)) throw new ArgumentException("Имя сервиса не может быть пустым.");
            _serviceName = value;
        }
    }

    public decimal Cost
    {
        get => _cost;
        set
        {
            if (value < 0) throw new ArgumentException("Стоимость не может быть отрицательной.");
            _cost = value;
        }
    }

    public Subscription(int id, string name, decimal cost)
    {
        SubscriptionId=id;
        ServiceName = name;
        Cost = cost;
    }

    public virtual decimal CalculateMonthlyCost()
    {
        return Cost;
    }
    public virtual void ExtendSubscription(int months)
    {
         Console.WriteLine($"Подписка '{ServiceName}' продлена на {months} мес. Стоимость продления: {Cost * months:C}");
    }
    public virtual void GetSubscriptionDetails()
    {
        Console.WriteLine($"ID: {SubscriptionId}, Услуга: {ServiceName}, Базовая стоимость: {Cost:C}/мес.");
    }
}

public class OnlineServiceSubscription : Subscription
{
    private int _maxUsers;

    public int MaxUsers
    {
        get => _maxUsers;
        set
        {
            if (value <= 0) throw new ArgumentException("Количество пользователей должно быть больше нуля.");
            _maxUsers = value;
        }
    }

    public OnlineServiceSubscription(int id, string name, decimal cost, int maxUsers): base(id, name, cost)
    {
        MaxUsers=maxUsers;
    }

    public override decimal CalculateMonthlyCost()
    {
        decimal extra = (MaxUsers > 1) ? (MaxUsers - 1) * Cost * 0.10m : 0;
        return Cost + extra;
    }

    public void ShowUserLimit()
    {
        Console.WriteLine($"Максимальное кол-во пользователей: {MaxUsers}");
    }
}

public class StreamingSubscription: Subscription
{
    private int _maxStreams;

    public int MaxStreams
    {
        get => _maxStreams;
        set
        {
            if (value <= 0) throw new ArgumentException("Количество потоков должно быть больше нуля.");
            _maxStreams = value;
        }
    }

    public StreamingSubscription(int id, string name, decimal cost, int maxStreams)
        : base(id, name, cost)
    {
        MaxStreams = maxStreams;
    }

    public override void ExtendSubscription(int months)
    {
        decimal discount = 0;
        if (months >= 6) discount = 0.20m;
        else if (months >= 3) discount = 0.10m;

        decimal total = Cost * months * (1 - discount);
        Console.WriteLine($"Продление стриминг-подписки '{ServiceName}' на {months} мес.");
        Console.WriteLine($"Скидка: {discount * 100}%. Итоговая стоимость: {total:C}");
        if (MaxStreams >= 3)
            Console.WriteLine("Бонус: +1 месяц бесплатно (акция для многопоточных подписок)!");
    }

    public void ShowStreamLimit()
    {
        Console.WriteLine($"Максимум одновременных потоков: {MaxStreams}");
    }
}

class VideoSubscription: Subscription
{
    private string _videoQuality;

    public string VideoQuality
    { 
        get => _videoQuality; 
        set
        {
            if (string.IsNullOrWhiteSpace(value)) throw new ArgumentException("Качество видео не может быть пустым.");
            _videoQuality = value;
        } 
    } 

    public VideoSubscription(int id, string name, decimal cost, string quality)
        : base(id, name, cost)
    {
        VideoQuality = quality;
    }

    public override void GetSubscriptionDetails()
    {
        Console.WriteLine($"ID: {SubscriptionId}, Услуга: {ServiceName}, Стоимость: {Cost:C}/мес., Качество видео: {VideoQuality}");
    }

    public void ChangeQuality(string newQuality)
    {
        VideoQuality = newQuality;
        Console.WriteLine($"Качество видео изменено на {newQuality}");
    }
}

public class SubscriptionHub
{
    public void CrossServiceInteraction(Subscription sub1, Subscription sub2)
    {
        Console.WriteLine($"\n--- Взаимодействие между '{sub1.ServiceName}' и '{sub2.ServiceName}' ---");

        if (sub1 is StreamingSubscription streaming && sub2 is VideoSubscription video)
        {
            Console.WriteLine($"Обнаружена совместимость пакетов!");
            if (streaming.MaxStreams >= 3 && video.VideoQuality != "4K")
            {
                Console.WriteLine($"Бонус за кросс-подписку: Качество '{video.ServiceName}' бесплатно улучшено до 4K!");
                video.ChangeQuality("4K");
            }
        }
        else
        {
            Console.WriteLine("Особых условий для совместного использованияподписок не найдено");
        }
    }
}

class Program
{
    public static void Main()
    {
        var onlineSub = new OnlineServiceSubscription(1, "Office 365", 10.00m, 5);
            var streamSub = new StreamingSubscription(2, "Netflix", 15.00m, 4);
            var videoSub = new VideoSubscription(3, "YouTube Premium", 12.00m, "FullHD");

            Subscription[] subscriptions = { onlineSub, streamSub, videoSub };

            Console.WriteLine("=== Демонстрация полиморфизма ===");
            foreach (var sub in subscriptions)
            {
                sub.GetSubscriptionDetails();
                Console.WriteLine($"Ежемесячная стоимость: {sub.CalculateMonthlyCost():C}");
                sub.ExtendSubscription(6);
                Console.WriteLine(new string('-', 40));
            }

            SubscriptionHub hub = new SubscriptionHub();
            hub.CrossServiceInteraction(streamSub, videoSub);
    }
}
Program.Main();



=== Демонстрация полиморфизма ===
ID: 1, Услуга: Office 365, Базовая стоимость: ¤10.00/мес.
Ежемесячная стоимость: ¤14.00
Подписка 'Office 365' продлена на 6 мес. Стоимость продления: ¤60.00
----------------------------------------
ID: 2, Услуга: Netflix, Базовая стоимость: ¤15.00/мес.
Ежемесячная стоимость: ¤15.00
Продление стриминг-подписки 'Netflix' на 6 мес.
Скидка: 20.00%. Итоговая стоимость: ¤72.00
Бонус: +1 месяц бесплатно (акция для многопоточных подписок)!
----------------------------------------
ID: 3, Услуга: YouTube Premium, Стоимость: ¤12.00/мес., Качество видео: FullHD
Ежемесячная стоимость: ¤12.00
Подписка 'YouTube Premium' продлена на 6 мес. Стоимость продления: ¤72.00
----------------------------------------

--- Взаимодействие между 'Netflix' и 'YouTube Premium' ---
Обнаружена совместимость пакетов!
Бонус за кросс-подписку: Качество 'YouTube Premium' бесплатно улучшено до 4K!
Качество видео изменено на 4K
